# MindFlow Step 1a · Daily JKX replication

**Question.** The chart-image signal of Jiang, Kelly & Xiu (2023) is daily (images of 5 or 20 days, labels over the next 5 or 20 days). Is it there on our stocks and years?


Same images, network and training rule as JKX; only the universe (large caps) and years differ.

| | JKX (2023) | here |
|---|---|---|
| data | CRSP daily, all US stocks, 1993–2019 | daily bars built from our 5-minute bars (open of the first bar, high, low, 16:00 close, summed volume) |
| images | I5 (32×15), I20 (64×60): OHLC, moving average, volume | same |
| labels | sign of the next 5 / 20-day return (`dir`) | same, plus the cross-sectional version (`xsdir`) |
| CNN | 2 / 3 blocks, 64-128-256, 5×3 conv, dropout 0.5, Adam 1e-5, batch 128 | same (I5 155,138 · I20 708,866 parameters) |
| training | random 70/30 split of the training years, early stop after 2 bad epochs, 5 seeds averaged | same |

Benchmarks on identical rows: HGB on daily hand features and the classic rules (1-day and 1-week reversal, 1- and
3-month momentum). **Leakage controls**: each window is divided by its anchor day's multipliers (point in time);
labels are h-day total returns; training labels end inside the training years (purge); validation and test anchors
start h sessions after a boundary (embargo); evaluation uses one anchor every h days (non-overlapping labels).


<details><summary>Where this notebook sits in the MindFlow</summary>


| step | question | notebooks |
|---|---|---|
| **1** | Does the JKX chart signal exist at the daily frequency on our large caps (also for pretrained models)? | `Step1a_Daily_JKX_CNN`, `Step1b_Daily_Pretrained` |
| **2** | Moved to 5-minute charts: does any model (CNN, ResNet, 1D CNN, ViT, HGB, frozen DINOv2/CLIP, time-series foundation models) find direction? | `Step2a_Intraday_Direction_Models`, `Step2b_Intraday_Direction_Pretrained` |
| **3** | Same models, target = volatility: is there a signal, and who wins? | `Step3a_Intraday_Volatility_Models`, `Step3b_Intraday_Volatility_Pretrained` |
| **4** | Add information to the picture step by step until the image model reaches HGB; is the image still useful beyond the features; which features matter? | `Step4a_Add_Image_Information`, `Step4b_Feature_Selection` |
| **5** | One figure and one parameter table over all steps | `Step5_Summary` |
| **6** | Next questions: does market-regime information (VIX, multi-day realized volatility) help? | `Step6a_VIX_Regime`, `Step6b_Final_Model` (final model + held-out test), `Step6c_Volatility_Use` (IC, long-short, risk sizing), `Step6d_Pure_Image` (the numbers drawn into the picture, no numbers input) |
| **7** | Efficient computing for the final model: 16-bit / INT8 inference, LoRA fine-tuning of a pretrained ViT | `Step7a_Efficient_Computing`, `Step7b_Distillation` (small students taught by the final CNN) |

</details>

**Data.** `alldata_bundle_v4.zip` (TAQ filter v4, point-in-time price adjustment): S&P 500 ∪ NASDAQ 100 ∪ DJIA 30
members by date, 2,514 sessions 2016-01-04 … 2025-12-31, five-minute bars plus SPY on the same clock.
**Split.** Train 2016–2022 · validation 2023 (every choice) · test 2024–2025 (last section, run once at the end).
**Advisor rule.** Every model prints its parameter count; for HGB the count is tree nodes.

## 1. Setup
Put `alldata_bundle_v4.zip` and the `code/` folder in `Demo/MindFlow` on Drive. The bundle (1.3 GB) is copied to the
Colab disk and unzipped once per runtime; the next notebook in the same runtime reuses it.

In [ ]:
import os, sys, json, shutil, time
from pathlib import Path

LOCAL_TEST = os.environ.get("MINDFLOW_LOCAL_TEST") == "1"          # repository QA only (synthetic data)
if LOCAL_TEST:
    HERE, LOCAL_ROOT = Path(os.environ["MINDFLOW_HERE"]), Path(os.environ["MINDFLOW_LOCAL"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    HERE, LOCAL_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo/MindFlow"), Path("/content/mindflow")
CODE = HERE / "code"
assert (CODE / "mindflow.py").is_file(), f"Put the code/ folder in {HERE}"
sys.path[:0] = [str(CODE)]
import mindflow as mf
PREPARED = mf.stage_bundle(HERE / mf.BUNDLE_NAME, LOCAL_ROOT)
PANELS_ADDED = mf.stage_panels(HERE, PREPARED)          # e.g. vixy_panel_v4.zip, if uploaded
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown
import daily_jkx as dj
print('daily_jkx', dj.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name() if DEVICE.type == "cuda" else "")

## 2. Configuration
**To resume after a disconnect**, set `RUN_ID` to the id printed by the first run and Run all again.
`QUICK = True`: one CNN instead of five, at most 60,000 training images, I20/R5 only (~10–15 min).
The full run (5 CNNs × 6 cells) takes roughly 1–2 hours on an L4 (images are small and stay on the GPU).
The JKX design is fixed in advance (nothing tuned on 2023), so `RUN_TEST = True` scores 2024–25 in the same run.

In [ ]:
SMOKE = LOCAL_TEST
assert DEVICE.type == "cuda" or SMOKE, "Select a GPU runtime"
RUN_ID = None                                       # None = new run; or an old RUN_ID to resume it
QUICK = False
RUN_TEST = True
CONFIGS = [(5, 5), (20, 5), (20, 20)]               # (image days, label horizon days): I5/R5, I20/R5, I20/R20
TARGETS = ["dir", "xsdir"]
SETTINGS = dict(train_stride=5, min_bars=60, seeds=5, learning_rate=1e-5, batch_size=128, max_epochs=30,
                patience=2, min_delta=0.0, dropout=0.5, holdout_share=0.3, max_train=None, seed=69)
if QUICK:
    SETTINGS.update(seeds=1, max_train=60_000); CONFIGS = [(20, 5)]
if SMOKE:
    SETTINGS.update(seeds=2, max_epochs=2, batch_size=64)
CELLS = [dict(name=f"{t}_I{L}R{h}", image_days=L, horizon=h, target=t) for L, h in CONFIGS for t in TARGETS]
RUN_ID, RUN_DIR, RESUME = mf.new_run(HERE, "1a", RUN_ID, "daily", LOCAL_TEST)
mf.save_settings(RUN_DIR, RESUME, settings=SETTINGS, cells=CELLS, quick=QUICK, run_test=RUN_TEST, daily_jkx=dj.__version__)
print("RUN_ID:", RUN_ID, "->", RUN_DIR)
print(len(CELLS), "cells:", ", ".join(c["name"] for c in CELLS))

## 3. Daily bars from the 5-minute bars
Sessions, share of stock-days present, and a check that the daily close equals the 16:00 bar's close.

In [ ]:
t0 = time.perf_counter()
store = dj.DailyStore(PREPARED, min_bars=SETTINGS["min_bars"])
cfg = store.config
print(f"{store.S} stocks x {store.D} sessions ({store.dates[0]} – {store.dates[-1]}) in {time.perf_counter() - t0:.0f}s")
years = pd.Series(store.dates).str[:4]
print("stock-days present by year:", (pd.Series(store.valid.mean(0)).groupby(years.values).mean().round(3)).to_dict())
print("splits: train ≤", cfg["train_end"], "| validation ≤", cfg["validation_end"], "| test after")
s0 = int(np.argmax(store.valid.sum(1)))
feats = np.load(PREPARED / "features.npy", mmap_mode="r")
last_bar_close = np.asarray(feats[s0, dj.SLOTS - 1::dj.SLOTS, 3], float)
ok = store.valid[s0]
assert np.allclose(store.ohlcv[s0, ok, 3], last_bar_close[ok]), "daily close is not the 16:00 bar's close"
print(f"stock {s0}: daily close equals the 16:00 bar close on all {ok.sum()} sessions")

## 4. Anchors and images
One anchor every 5 sessions for every stock; each image length is rendered once and shared by every cell.

In [ ]:
import matplotlib.pyplot as plt
FRAMES, IMAGES = {}, {}
for L in sorted({L for L, _ in CONFIGS}):
    t0 = time.perf_counter()
    FRAMES[L] = dj.anchors(store, L, SETTINGS["train_stride"])
    imgs = dj.render_all(store, FRAMES[L], L)
    IMAGES[L] = torch.from_numpy(imgs).to(DEVICE)
    print(f"I{L}: {len(FRAMES[L]):,} images {imgs.shape[1:]} in {time.perf_counter() - t0:.0f}s")
    ex = FRAMES[L].sample(4, random_state=1)
    fig, axes = plt.subplots(1, 4, figsize=(10, 3))
    for ax, (i, r) in zip(axes, ex.iterrows()):
        ax.imshow(imgs[i], cmap="gray", aspect="auto"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"I{L}  stock {r.security_index}  {r.date}", fontsize=8)
    fig.tight_layout(); fig.savefig(RUN_DIR / f"examples_I{L}.png", dpi=110); plt.show()
for L in IMAGES:
    m = dj.JKXCNN(L)
    print(f"JKX CNN I{L}: {dj.count(m):,} parameters (fully connected fan-in {m.flat:,})")

## 5. Train (CNN × seeds, HGB, rules) per cell
Finished cells are saved to `runs/step1a_daily_jkx/<RUN_ID>/<cell>/` and skipped on a rerun.

In [ ]:
RESULTS, t0 = {}, time.perf_counter()
for i, cell in enumerate(CELLS, 1):
    print("\n" + "=" * 20, f"[{i}/{len(CELLS)}]", cell["name"], "=" * 20)
    RESULTS[cell["name"]] = dj.run_cell(store, cell, FRAMES[cell["image_days"]], IMAGES[cell["image_days"]],
                                        SETTINGS, DEVICE, RUN_DIR / cell["name"])
    print(f"elapsed {(time.perf_counter() - t0) / 60:.1f} min")
if not RUN_TEST:
    for r in RESULTS.values():
        r["predictions"] = r["predictions"][r["predictions"].split != "test"]

## 6. Results
AUC with 95% intervals from resampling whole rebalance dates; accuracy and log-loss gain for the CNN and HGB;
`ls_*` = equal-weighted top-minus-bottom decile return over the next h days (bps per period, t-statistic, Sharpe).

In [ ]:
TABLE = dj.evaluate(RESULTS, n_boot=50 if SMOKE else 300)
TABLE.to_csv(RUN_DIR / "daily_table.csv", index=False)
cols = ["image", "horizon_days", "model", "auc", "auc_low", "auc_high", "accuracy", "log_loss_gain_pct",
        "ls_mean_bps", "ls_t", "ls_sharpe_ann", "dates", "rows"]
for split in ("validation", "test"):
    for tgt in TARGETS:
        t = TABLE[(TABLE.split == split) & (TABLE.target == tgt)]
        if len(t):
            print(f"{split} — {tgt}: {dj.TARGETS[tgt]}")
            display(t[[c for c in cols if c in t]].round(4))
PARAMS = pd.DataFrame([dict(cell=k, **v["info"]["parameters"]) for k, v in RESULTS.items()]).set_index("cell")
print("Parameters (CNN: weights; HGB: tree nodes):"); display(PARAMS)
TAKEAWAYS = dj.takeaways(TABLE)
(RUN_DIR / "takeaways.md").write_text(TAKEAWAYS + "\n"); display(Markdown(TAKEAWAYS))

### How to read it
- **CNN AUC clearly above 0.5 with a positive long-short**: the JKX signal exists here at the daily frequency.
- **≈ 0.50 everywhere** (what the v1 data gave): no daily image signal in these large caps. JKX find most of their
  effect in small stocks (equal-weighted Sharpe ≫ value-weighted), which this universe does not contain.
- Step 1b runs pretrained models on exactly these rows; Step 5 puts the daily and 5-minute AUCs side by side.